In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

In [2]:
# 24 hourly intervals
T = 24
hours = np.arange(T)

In [3]:
# Wind: random + mild daily shape
wind = 30 + 10 * np.sin(2 * np.pi * hours / 24 + 0.5) + np.random.normal(0, 3, T)
wind = np.clip(wind, 0, None)

# Solar: zero at night + bell curve at midday
solar = np.maximum(0, 50 * np.exp(-0.5 * ((hours - 12) / 4) ** 2))

# Demand: daily pattern (morning + evening peaks)
demand = 60 + 10 * np.sin(2 * np.pi * (hours - 7) / 24) + 5 * np.sin(4 * np.pi * (hours - 17) / 24)

In [4]:
# Day-ahead prices: typical Nordic pattern
price_DA = 40 + 15 * np.sin(2 * np.pi * (hours - 7) / 24) + np.random.normal(0, 2, T)

# Intraday prices: noisier & slightly higher volatility
price_ID = price_DA + np.random.normal(0, 5, T)

In [5]:
G_actual = wind + solar - demand

In [6]:
forecast_error_level = 0.15  # 15%

# Forecasts are actuals + noise
G_forecast = G_actual * (1 + np.random.normal(0, forecast_error_level, T))

In [7]:
alpha = 0.7  # risk appetite

q_DA = alpha * G_forecast

In [8]:
# imbalance
imbalance = G_actual - q_DA

# profit
profit_DA = q_DA * price_DA
profit_ID = imbalance * price_ID
profit_total = profit_DA + profit_ID

total_profit = profit_total.sum()

total_profit

np.float64(-5761.034978942243)

In [9]:
df = pd.DataFrame({
    "Hour": hours,
    "Wind": wind,
    "Solar": solar,
    "Demand": demand,
    "Net Actual": G_actual,
    "Net Forecast": G_forecast,
    "Commit DA": q_DA,
})

fig = go.Figure()
for col in ["Wind", "Solar", "Demand", "Net Actual", "Net Forecast", "Commit DA"]:
    fig.add_trace(go.Scatter(x=df["Hour"], y=df[col], mode="lines", name=col))

fig.update_layout(title="Production, Forecast, Commitments")
fig.show()

In [10]:
fig = go.Figure()
fig.add_trace(go.Bar(x=hours, y=profit_total))
fig.update_layout(title="Profit per Hour (EUR)", xaxis_title="Hour", yaxis_title="Profit")
fig.show()